# Drift Colab Demo

Minimal Colab demo for direct sample generation and visualization.

Default example:
- `init_from = hf://latent_L_sota`
- `class_ids = 95,22,88,108,386,296,483,698`

Reference names for the default 8 classes:
- `95` jacamar
- `22` bald eagle
- `88` macaw
- `108` sea anemone
- `386` African elephant
- `296` ice bear
- `483` castle
- `698` palace

In [ ]:
#@title 1. Setup
import os
from pathlib import Path

# ==========================================
# Clone Repository
# ==========================================
REPO_URL = "https://github.com/lambertae/drifting.git"
REPO_REF = "main"
REPO_DIR = Path("/content/drifting")

if not REPO_DIR.exists():
    print("Cloning repository...")
    get_ipython().system(f"git clone --branch {REPO_REF} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print("Repository already cloned.")

if REPO_DIR.exists():
    get_ipython().run_line_magic("cd", str(REPO_DIR))
else:
    print("Error: Repository directory not found.")

# ==========================================
# Environment & Dependencies
# ==========================================
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("HF_ROOT", "/content/hf_cache")
print("Installing dependencies...")
get_ipython().system("pip install -q -r requirements.txt")
print("Setup Complete! Public HF downloads do not require a token.")

In [ ]:
#@title 2. Helpers
import math
import sys

import jax
import matplotlib.pyplot as plt
import numpy as np

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from inference import _load_model
from utils.hsdp_util import set_global_mesh

MODEL_CACHE = {}
set_global_mesh(min(8, max(1, jax.local_device_count() * jax.process_count())))
print(f"JAX devices: {jax.devices()}")


def parse_class_ids(text: str) -> list[int]:
    class_ids = [int(part.strip()) for part in text.split(",") if part.strip()]
    if not class_ids:
        raise ValueError("Provide at least one class index.")
    for class_id in class_ids:
        if not 0 <= class_id < 1000:
            raise ValueError(f"class id out of range: {class_id}")
    return class_ids


def load_pipeline(init_from: str):
    init_from = init_from.strip()
    if not init_from:
        raise ValueError("init_from is empty.")
    if init_from not in MODEL_CACHE:
        print(f"Loading model from {init_from} ...")
        MODEL_CACHE[init_from] = _load_model(init_from)
    return MODEL_CACHE[init_from]


def generate_samples(gen_step_jit, params, *, class_ids: list[int], cfg_scale: float, seed: int) -> np.ndarray:
    labels = np.asarray(class_ids, dtype=np.int32)
    samples = []
    batch_size = min(4, len(labels))
    for start in range(0, len(labels), batch_size):
        batch_labels = labels[start : start + batch_size]
        batch = (np.zeros((len(batch_labels), 1), dtype=np.int32), batch_labels)
        out = gen_step_jit(
            batch,
            params=params,
            cfg_scale=cfg_scale,
            rng=jax.random.PRNGKey(seed + start),
        )
        samples.append(np.asarray(jax.device_get(out), dtype=np.float32))
    return np.concatenate(samples, axis=0)


def show_samples(samples: np.ndarray, class_ids: list[int]) -> None:
    images = np.asarray(samples)
    if images.shape[1] in (3, 4):
        images = images.transpose(0, 2, 3, 1)
    images = np.clip(images[..., :3], 0.0, 1.0)

    cols = min(4, len(images))
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for idx, ax in enumerate(axes.flat):
        ax.axis("off")
        if idx >= len(images):
            continue
        ax.imshow(images[idx])
        ax.set_title(f"class {class_ids[idx]:03d}", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
#@title 3. Generate
init_from = "hf://latent_L_sota" #@param {type:"string"}
class_ids_text = "95,22,88,108,386,296,483,698" #@param {type:"string"}
cfg_scale = 2.5 #@param {type:"number"}
seed = 0 #@param {type:"integer"}

class_ids = parse_class_ids(class_ids_text)
gen_step_jit, params, metadata = load_pipeline(init_from)
samples = generate_samples(
    gen_step_jit,
    params,
    class_ids=class_ids,
    cfg_scale=float(cfg_scale),
    seed=int(seed),
)
print(f"Generated samples: shape={samples.shape}, dtype={samples.dtype}")
print(f"Model in_channels: {metadata.get('model_config', {}).get('in_channels')}")
show_samples(samples, class_ids)
